# FASE 3: YOLO26n + Coordinate Attention + Edge Contour (Google Colab GPU T4)
## Deteksi Penyakit Daun Teh — Perbandingan Baseline vs CA & Kontur Daun Utuh

Notebook ini melatih **YOLO26n + Coordinate Attention (CA)**, membandingkan
hasilnya dengan **YOLO26n Baseline** (Fase 2), serta mengimplementasikan
**Edge Contour Detection** untuk membentuk deteksi mengikuti kontur daun dan memfilter daun tidak utuh.

### Modifikasi & Fitur yang Dilakukan:
- **3 blok CoordAttYOLO** disisipkan di backbone P3, P4, P5
- Hyperparameter identik dengan baseline (50 epoch, seed=42, imgsz=640)
- Pretrained weights `yolo26n.pt` tetap digunakan (layer CA baru diinisialisasi random)
- **Edge Contour Post-processing (Canny + Morfologi)**: bounding box diubah menjadi polygon kontur daun asli
- **Incomplete Leaf Suppression**: menyaring dan mengabaikan daun yang terpotong batas gambar/ROI

### Referensi Paper:
> Hou, Q., Zhou, D., & Feng, J. (2021). **Coordinate Attention for Efficient Mobile Network Design**. *CVPR 2021*. https://arxiv.org/abs/2103.02907

---
> **Estimasi waktu**: ~8–12 menit di GPU T4 (50 epoch, imgsz=640, batch=16)

### Langkah 1: Pastikan GPU T4 Aktif
Klik **Runtime** → **Change runtime type** → pilih **T4 GPU** → **Save**.

In [ ]:
!nvidia-smi

### Langkah 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Langkah 3: Upload ZIP ke Google Drive (lakukan sekali)

Upload berkas berikut ke **Google Drive > SKRIPSI/**:
1. `tea_yolo.zip` — dataset gambar & anotasi
2. `phase3_scripts.zip` — kode Fase 3 (CA module, YAML, edge contour, test external)
3. `yolo26n.pt` — pretrained weights

Untuk membuat `phase3_scripts.zip`, jalankan di PowerShell lokal:
```powershell
Compress-Archive -Path D:\SKRIPSI\phase3_scripts -DestinationPath D:\SKRIPSI\phase3_scripts.zip -Force
```

### Langkah 4: Ekstrak Dataset & Kode ke Disk Lokal Colab (NVMe SSD)

In [ ]:
# Salin dari Drive ke disk lokal Colab (jauh lebih cepat dibaca)
!cp /content/drive/MyDrive/SKRIPSI/tea_yolo.zip       /content/
!cp /content/drive/MyDrive/SKRIPSI/phase3_scripts.zip /content/
!cp /content/drive/MyDrive/SKRIPSI/yolo26n.pt          /content/

# Ekstrak
!unzip -q /content/tea_yolo.zip       -d /content/
!unzip -q /content/phase3_scripts.zip -d /content/

# Verifikasi
!echo "=== Dataset ==="
!ls /content/tea_yolo/
!echo "=== Phase3 Scripts ==="
!ls /content/phase3_scripts/
!echo "=== yolo26n.pt ==="
!ls -lh /content/yolo26n.pt

### Langkah 5: Install Dependencies

In [ ]:
!pip install -q ultralytics opencv-python-headless

import torch
print("PyTorch  :", torch.__version__)
print("CUDA OK  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU      :", torch.cuda.get_device_name(0))
    print("VRAM     :", round(torch.cuda.get_device_properties(0).total_memory/1e9,2), "GB")

### Langkah 6: Verifikasi Modul CA & Edge Contour (Unit Test)
Pastikan modul CoordAtt dan EdgeContour berjalan lancar sebelum training.

In [ ]:
import sys
sys.path.insert(0, '/content')

# 1. Unit test modul CA
!python /content/phase3_scripts/ca_module.py

# 2. Unit test modul Edge Contour
!python /content/phase3_scripts/edge_contour.py

---
## 🚀 TRAINING: YOLO26n + Coordinate Attention (50 Epoch, GPU T4, 640×640)

Estimasi waktu: **~8–12 menit** di T4.

In [ ]:
import sys
sys.path.insert(0, '/content')

# Training YOLO26n + CA
!python /content/phase3_scripts/01_train_ca.py \
    --device cuda \
    --epochs 50 \
    --imgsz 640 \
    --batch 16 \
    --workers 8 \
    --run_name yolo26n_ca_50ep

---
## 📊 EVALUASI: Test Set (529 Gambar)

Evaluasi model terbaik pada test set yang tidak pernah dilihat saat training.

In [ ]:
!python /content/phase3_scripts/02_evaluate_testset.py \
    --device cuda \
    --imgsz 640 \
    --batch 16 \
    --workers 8 \
    --run_name yolo26n_ca_50ep

---
## 🔍 ANALISIS VISUAL: TP, FP, WC, FN

In [ ]:
!python /content/phase3_scripts/03_visual_analysis.py \
    --device cuda \
    --run_name yolo26n_ca_50ep \
    --conf 0.25 \
    --iou_thr 0.5 \
    --max_samples 10

---
## 📑 PERBANDINGAN: Baseline vs CA

Perbandingan lengkap Baseline (Fase 2) vs CA (Fase 3).

> **Catatan**: Salin dulu hasil baseline Fase 2 dari Drive ke `/content/phase2_runs/`

In [ ]:
# Salin hasil Baseline Fase 2 dari Drive ke Colab untuk perbandingan
!mkdir -p /content/phase2_runs/yolo26n_baseline_50ep/test_eval/test_results
!mkdir -p /content/phase2_runs/yolo26n_baseline_50ep/visual_analysis

!cp /content/drive/MyDrive/SKRIPSI/phase2_runs/yolo26n_baseline_50ep/test_eval/test_results/summary_metrics.json \
   /content/phase2_runs/yolo26n_baseline_50ep/test_eval/test_results/ 2>/dev/null || \
   echo '[INFO] summary_metrics.json baseline tidak ditemukan — pastikan file ada di Drive.'

!cp /content/drive/MyDrive/SKRIPSI/phase2_runs/yolo26n_baseline_50ep/visual_analysis/visual_stats.json \
   /content/phase2_runs/yolo26n_baseline_50ep/visual_analysis/ 2>/dev/null || echo '[INFO] visual_stats.json tidak ditemukan.'

In [ ]:
!mkdir -p /content/laporan

!python /content/phase3_scripts/04_compare_baseline_vs_ca.py \
    --run_name_ca yolo26n_ca_50ep \
    --run_name_baseline yolo26n_baseline_50ep

### Tampilkan Laporan Perbandingan

In [ ]:
from IPython.display import display, Markdown
with open('/content/laporan/fase3_ca_vs_baseline.md', 'r', encoding='utf-8') as f:
    display(Markdown(f.read()))

---
## 🌿 POST-PROCESSING: Edge Detection & Kontur Daun (Incomplete Leaf Filter)

Modul ini mengubah bounding box YOLO menjadi **POLYGON KONTUR DAUN** mengikuti bentuk daun asli,
serta **memfilter daun yang tidak utuh** (terpotong oleh tepi foto / ROI).

### Fitur Utama:
1. **Canny Edge Detection + Morfologi Closing**: Mengikuti lekukan daun meskipun tepi tidak sempurna/rusak oleh penyakit.
2. **Incomplete Leaf Suppression**: Deteksi yang menyentuh batas tepi dieliminasi secara otomatis.
3. **Visualisasi Dual**: Menampilkan polygon kontur daun utuh, serta menandai daun yang di-suppress (merah).

In [ ]:
# Jalankan inferensi dengan edge contour pada Test Set (contoh 20 gambar)
!python /content/phase3_scripts/05_infer_with_contour.py \
    --source /content/tea_yolo/images/test \
    --weights /content/phase3_runs/yolo26n_ca_50ep/weights/best.pt \
    --device cuda \
    --conf 0.25 \
    --max_images 20 \
    --save \
    --save_stats \
    --show_suppressed

### Visualisasikan Hasil Deteksi Kontur Daun di Notebook

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

output_images = sorted(glob.glob('/content/phase3_runs/contour_inference/*.jpg'))[:6]
if output_images:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    for ax, p in zip(axes.flatten(), output_images):
        ax.imshow(Image.open(p))
        ax.set_title(p.split('/')[-1], fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Belum ada gambar output atau tidak ada deteksi tersimpan.')

---
## 🧪 TESTING GAMBAR DARI GOOGLE / INTERNET / UPLOAD SENDIRI

Uji hasil training model dengan gambar daun teh dari Google Images atau foto baru Anda sendiri.

### Opsi A: Test menggunakan URL gambar langsung dari Google / Internet

In [ ]:
# Masukkan link URL gambar daun teh dari Google / internet di bawah ini:
IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/48/Camellia_sinensis_-_Köhler–s_Medizinal-Pflanzen-025.jpg/800px-Camellia_sinensis_-_Köhler–s_Medizinal-Pflanzen-025.jpg"

!python /content/phase3_scripts/06_test_external_image.py \
    --url "$IMAGE_URL" \
    --weights /content/phase3_runs/yolo26n_ca_50ep/weights/best.pt \
    --conf 0.25

# Tampilkan hasil visualisasi 4 panel (Asli vs YOLO vs Kontur Daun vs Canny Edge)
from IPython.display import Image, display
display(Image("test_google_results/hasil_testing_lengkap.png"))

### Opsi B: Upload file gambar dari Komputer / HP Anda langsung

In [ ]:
from google.colab import files
print("Silakan pilih file gambar daun dari laptop/HP Anda:")
uploaded = files.upload()

for fn in uploaded.keys():
    print(f"\nMemproses gambar: {fn}...")
    !python /content/phase3_scripts/06_test_external_image.py \
        --image "{fn}" \
        --weights /content/phase3_runs/yolo26n_ca_50ep/weights/best.pt \
        --conf 0.25
    
    from IPython.display import Image, display
    display(Image("test_google_results/hasil_testing_lengkap.png"))

---
## 💾 SIMPAN SEMUA HASIL KE GOOGLE DRIVE

In [ ]:
# Buat folder tujuan di Drive
!mkdir -p /content/drive/MyDrive/SKRIPSI/phase3_runs/yolo26n_ca_50ep
!mkdir -p /content/drive/MyDrive/SKRIPSI/phase3_runs/contour_inference
!mkdir -p /content/drive/MyDrive/SKRIPSI/phase3_runs/test_google_results
!mkdir -p /content/drive/MyDrive/SKRIPSI/laporan

# Salin seluruh hasil training & evaluasi CA
!cp -r /content/phase3_runs/yolo26n_ca_50ep/* \
        /content/drive/MyDrive/SKRIPSI/phase3_runs/yolo26n_ca_50ep/

# Salin hasil contour inference jika ada
!cp -r /content/phase3_runs/contour_inference/* \
        /content/drive/MyDrive/SKRIPSI/phase3_runs/contour_inference/ 2>/dev/null || true

# Salin hasil testing google jika ada
!cp -r test_google_results/* \
        /content/drive/MyDrive/SKRIPSI/phase3_runs/test_google_results/ 2>/dev/null || true

# Salin laporan perbandingan
!cp /content/laporan/fase3_ca_vs_baseline.md \
    /content/drive/MyDrive/SKRIPSI/laporan/

print("")
print("✅ SEMUA HASIL TERSIMPAN KE GOOGLE DRIVE!")
print("   Drive > SKRIPSI > phase3_runs > yolo26n_ca_50ep/")
print("   Drive > SKRIPSI > phase3_runs > contour_inference/")
print("   Drive > SKRIPSI > phase3_runs > test_google_results/")
print("   Drive > SKRIPSI > laporan > fase3_ca_vs_baseline.md")
print("")
print("File yang tersimpan:")
!ls /content/drive/MyDrive/SKRIPSI/phase3_runs/yolo26n_ca_50ep/
print("")
print("Best weights CA:")
!ls -lh /content/drive/MyDrive/SKRIPSI/phase3_runs/yolo26n_ca_50ep/weights/